# 04 Interests: what a student enjoys → careers → fields of study

Builds `data/processed/interest_links.csv` for the "I'm not sure yet" path. Run after `03_export.ipynb` (it reads `programs.csv`).

**Inputs** (not in GitHub; put them in `data/raw/`):
- O*NET 31.0 database, Excel format, unzipped to `data/raw/db_31_0_excel/` — https://www.onetcenter.org/database.html (U.S. Department of Labor, CC BY 4.0). Files used: `Career Interest Types.xlsx`, `Job Zones.xlsx`.
- NCES/BLS CIP 2020 – SOC 2018 crosswalk: `data/raw/CIP2020_SOC2018_Crosswalk.xlsx` — https://nces.ed.gov/ipeds/cipcode/resources.aspx?y=56

**Rules** (agreed design decisions)
1. Interest → career: each career's *first* career interest type (RIASEC) in O*NET.
2. Careers that usually need training, an associate degree, a bachelor's degree or graduate school (Job Zones 3–5). Zone 5 is shown with the label "Usually needs graduate school".
3. One row per 6-digit SOC (the crosswalk's level); postsecondary teachers (25-1xxx) are left out because they link to almost every field.
4. Career → program: the crosswalk (SOC-CIP sheet), rolled up to the 4-digit CIP codes used in `programs.csv`.
5. `CIP6_LINKS` = how many detailed (6-digit) programs in the field link to the career; the app shows the most closely tied careers first.
6. Liberal Arts (CIP 24) has no career links in the crosswalk, so the app shows it as a separate "Still deciding?" option.


In [ ]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
ONET = RAW / "db_31_0_excel"
OUT = Path("../data/processed")


In [ ]:
# 1. O*NET: first career interest type (RIASEC) and Job Zone of each occupation
TYPES = {1: "R", 2: "I", 3: "A", 4: "S", 5: "E", 6: "C"}
ci = pd.read_excel(ONET / "Career Interest Types.xlsx")
first = ci[ci["Element Name"] == "First Interest High-Point"][["O*NET-SOC Code", "Data Value"]]
first["INTEREST"] = first["Data Value"].round().astype(int).map(TYPES)
zones = pd.read_excel(ONET / "Job Zones.xlsx")[["O*NET-SOC Code", "Title", "Job Zone"]]
occ = zones.merge(first[["O*NET-SOC Code", "INTEREST"]], on="O*NET-SOC Code")
occ = occ[occ["Job Zone"].isin([3, 4, 5])]          # associate / bachelor's / graduate school
occ["SOC"] = occ["O*NET-SOC Code"].str[:7]


In [ ]:
# One row per 6-digit SOC (the crosswalk's level); keep the broad ".00" title when there is one
occ = occ.sort_values("O*NET-SOC Code").groupby("SOC", as_index=False).first()
occ = occ[~occ["SOC"].str.startswith("25-1")]       # postsecondary teachers link to almost every field
print("Occupations kept:", len(occ), occ["Job Zone"].value_counts().sort_index().to_dict())


In [ ]:
# 2. NCES/BLS crosswalk: occupation (SOC 2018) -> program (CIP 2020)
cw = pd.read_excel(RAW / "CIP2020_SOC2018_Crosswalk.xlsx", sheet_name="SOC-CIP", dtype=str)
cw = cw[cw["CIP2020Code"] != "99.9999"]
cw["CIP6"] = cw["CIP2020Code"]
cw["CIP4"] = cw["CIP6"].str.replace(".", "", regex=False).str[:4]
links = occ.merge(cw[["SOC2018Code", "CIP6", "CIP4"]], left_on="SOC", right_on="SOC2018Code")
links["FAMILY"] = links["CIP4"].str[:2]


In [ ]:
# 3. Keep programs that exist in the Scorecard program file
programs = pd.read_csv(OUT / "programs.csv", dtype=str)
known = set(programs["CIPCODE"].str.zfill(4))
links = links[links["CIP4"].isin(known)]


In [ ]:
# How closely a career is tied to a field: number of 6-digit programs in that field linked to it
links["CIP6_LINKS"] = links.groupby(["SOC", "FAMILY"])["CIP6"].transform("nunique")
out = (links[["INTEREST", "FAMILY", "SOC", "Title", "Job Zone", "CIP6_LINKS", "CIP4"]]
       .drop_duplicates()
       .rename(columns={"Title": "TITLE", "Job Zone": "JOB_ZONE"})
       .sort_values(["INTEREST", "FAMILY", "SOC", "CIP4"]))
out.to_csv(OUT / "interest_links.csv", index=False)
print("interest_links.csv:", out.shape)
print(out.groupby("INTEREST")["FAMILY"].nunique().to_dict())
print("Liberal Arts (24) present:", (out["FAMILY"] == "24").any())
